In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from pyspark.sql import functions as F

from src.config import load_config
from src.spark_session import get_spark
from src.evaluation.metrics import per_impression_metrics, evaluate

cfg = load_config("small")
spark = get_spark(cfg)
OUT = cfg["paths"]["processed"]

imp = spark.read.parquet(str(OUT / "impressions_long"))
test = imp.filter("periodo = 'test'")

In [2]:
def mrr_ref(y, s):
    order = np.argsort(s)[::-1]
    y = np.take(y, order)
    return np.sum(y / (np.arange(len(y)) + 1)) / np.sum(y)

def dcg_ref(y, s, k):
    order = np.argsort(s)[::-1]
    y = np.take(y, order[:k])
    return np.sum((2 ** y - 1) / np.log2(np.arange(len(y)) + 2))

def ndcg_ref(y, s, k):
    return dcg_ref(y, s, k) / dcg_ref(y, y, k)

def metricas_ref(g):
    y, s = g["label"].to_numpy(), g["score"].to_numpy()
    return pd.Series({
        "auc_ref": roc_auc_score(y, s) if 0 < y.sum() < len(y) else np.nan,
        "mrr_ref": mrr_ref(y, s),
        "ndcg5_ref": ndcg_ref(y, s, 5),
        "ndcg10_ref": ndcg_ref(y, s, 10),
    })

In [3]:
# Muestra de ~3 % de las impresiones de test, con puntajes continuos aleatorios
ids = test.select("impression_id").distinct().sample(fraction=0.03, seed=1)
muestra = (test.join(ids, "impression_id")
               .withColumn("score", F.rand(7))
               .select("impression_id", "label", "score")
               .cache())

ref = muestra.toPandas().groupby("impression_id").apply(metricas_ref).reset_index()
spk = per_impression_metrics(muestra).toPandas()
cmp = spk.merge(ref, on="impression_id")

print("Impresiones comparadas:", len(cmp))
for m in ["auc", "mrr", "ndcg5", "ndcg10"]:
    print(f"{m:7s} diferencia máxima = {np.nanmax(np.abs(cmp[m] - cmp[m + '_ref'])):.2e}")

Impresiones comparadas: 2143
auc     diferencia máxima = 1.11e-16
mrr     diferencia máxima = 2.78e-17
ndcg5   diferencia máxima = 0.00e+00
ndcg10  diferencia máxima = 2.22e-16


In [4]:
# Puntajes con solo 3 valores posibles -> empates masivos, como en la popularidad
muestra_emp = muestra.withColumn("score", F.floor(F.rand(11) * 3)).cache()

ref_e = muestra_emp.toPandas().groupby("impression_id").apply(metricas_ref).reset_index()
spk_e = per_impression_metrics(muestra_emp).toPandas()
cmp_e = spk_e.merge(ref_e, on="impression_id")

print(f"AUC con empates, diferencia máxima = {np.nanmax(np.abs(cmp_e['auc'] - cmp_e['auc_ref'])):.2e}")
print(f"MRR con empates, diferencia media  = {np.nanmean(np.abs(cmp_e['mrr'] - cmp_e['mrr_ref'])):.3f}  (esperada: distintos desempates)")

AUC con empates, diferencia máxima = 1.11e-16
MRR con empates, diferencia media  = 0.093  (esperada: distintos desempates)


In [5]:
for seed in [1, 2, 3]:
    _, res = evaluate(test.withColumn("score", F.rand(seed)), seed=seed)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in res.items()})

{'impresiones': 73152, 'impresiones_sin_auc': 0, 'auc': 0.4971, 'mrr': 0.2166, 'ndcg5': 0.2212, 'ndcg10': 0.2843}
{'impresiones': 73152, 'impresiones_sin_auc': 0, 'auc': 0.4992, 'mrr': 0.2179, 'ndcg5': 0.2232, 'ndcg10': 0.2857}
{'impresiones': 73152, 'impresiones_sin_auc': 0, 'auc': 0.5001, 'mrr': 0.2187, 'ndcg5': 0.224, 'ndcg10': 0.2865}


In [6]:
spark.stop()